In [17]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train = df_train.drop("Type", axis=1)
df_test = df_test.drop("Type", axis=1)

df_train.head()

,SampleID,Title,Description,Year,Score
0,tt9310400,The Stranded,When a tsunami strands dozens of teens on an i...,2019,6.2
1,tt5520732,Good Morning Call,A high school girl finally gets her own apartm...,2016,7.7
2,tt8499662,The Could've-Gone-All-the-Way Committee,People bring their stories of missed lust conn...,2018,6.8
3,tt16583602,Midnight Asia: Eat · Dance · Dream,Night brings out the most playful side of Asia...,2022,6.6
4,tt12867810,Start-Up,Young entrepreneurs aspiring to launch virtual...,2020,8.1


In [19]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

def preprocessing(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'[\d+]', '', text)
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return ' '.join(tokens)

df_train.Description = df_train.Description.apply(preprocessing)
df_test.Description = df_test.Description.apply(preprocessing)

df_train.Title = df_train.Title.apply(preprocessing)
df_test.Title = df_test.Title.apply(preprocessing)

df_train.head()

,SampleID,Title,Description,Year,Score
0,tt9310400,strand,tsunami strand dozen teen island private schoo...,2019,6.2
1,tt5520732,good morning,high school girl finally apartment share popul...,2016,7.7
2,tt8499662,couldvegonealltheway committee,people bring story miss lust connection object...,2018,6.8
3,tt16583602,midnight asia eat dance dream,night bring playful asias metropolis docuserie...,2022,6.6
4,tt12867810,startup,young entrepreneur aspire launch virtual dream...,2020,8.1


In [69]:
#preprocessing
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_year = scaler.fit_transform(df_train[['Year']])
X_final_year = scaler.transform(df_test[['Year']])

from sklearn.feature_extraction.text import TfidfVectorizer

all_text = pd.concat([df_train['Description'], df_test['Description']])
vect = TfidfVectorizer()
vect.fit(all_text)

X_train_text = vect.transform(df_train['Description'])
X_final_text = vect.transform(df_test['Description'])

all_titles = pd.concat([df_train['Title'], df_test['Title']])
vect2 = TfidfVectorizer()
vect2.fit(all_titles)

X_train_title = vect2.transform(df_train['Title'])
X_final_title = vect2.transform(df_test['Title'])

from scipy.sparse import hstack

X = hstack([X_train_year, X_train_text, X_train_title])
X_final = hstack([X_final_year, X_final_text, X_final_title])

y = df_train.Score

X

<1277x10272 sparse matrix of type '<class 'numpy.float64'>'
	with 27536 stored elements in COOrdinate format>

In [70]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import cross_val_score

model = ElasticNet()

cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')

array([-0.85710586, -0.80784693, -0.74498561, -0.84210583, -0.82808334])

In [71]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [72]:
submission = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "Score": y_preds
})

submission

,SampleID,Score
0,tt13321232,7.041269
1,tt10487072,7.041269
2,tt8004578,7.041269
3,tt11822998,7.041269
4,tt10348700,7.041269
...,...,...
315,tt9581782,7.041269
316,tt7971476,7.041269
317,tt6626676,7.041269
318,tt5545584,7.041269


In [73]:
submission.to_csv("submission.csv", index=False)